In [1]:
import time
import random

def read_dimacs(filename):
    graph = []
    with open(filename, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or line[0] == 'c':
                continue
            parts = line.split()
            if parts[0] == 'p':
                n = int(parts[2])
                graph = [set() for _ in range(n)]
            elif parts[0] == 'e':
                u = int(parts[1]) - 1
                v = int(parts[2]) - 1
                graph[u].add(v)
                graph[v].add(u)
    return graph

def greedy_clique_from(graph, start):
    clique = {start}
    candidates = list(graph[start])
    random.shuffle(candidates)
    for v in candidates:
        if all(v in graph[u] for u in clique):
            clique.add(v)
            candidates = [w for w in candidates if w in graph[v]]
    return clique

def heuristic_max_clique(graph, iterations=100):
    n = len(graph)
    best_clique = set()
    vertices = list(range(n))
    random.shuffle(vertices)
    for v in vertices:
        clique = greedy_clique_from(graph, v)
        if len(clique) > len(best_clique):
            best_clique = clique
    for _ in range(iterations):
        v = random.choice(vertices)
        clique = greedy_clique_from(graph, v)
        if len(clique) > len(best_clique):
            best_clique = clique
    return best_clique

def greedy_coloring(graph, candidates):
    n = len(candidates)
    colors = [0] * n
    max_color = 0
    for i, v in enumerate(candidates):
        forbidden = set()
        for j in range(i):
            u = candidates[j]
            if u in graph[v]:
                forbidden.add(colors[j])
        col = 1
        while col in forbidden:
            col += 1
        colors[i] = col
        max_color = max(max_color, col)
    return max_color

def branch_and_bound(graph, candidates, current_clique, best_clique, start_time, time_limit):
    if time.time() - start_time >= time_limit:
        return best_clique
    if not candidates:
        if len(current_clique) > len(best_clique):
            return current_clique.copy()
        return best_clique

    color_bound = greedy_coloring(graph, candidates)
    if len(current_clique) + color_bound <= len(best_clique):
        return best_clique

    for i, v in enumerate(candidates):
        if time.time() - start_time >= time_limit:
            break
        new_clique = current_clique.copy()
        new_clique.add(v)
        new_candidates = [w for w in candidates[i+1:] if w in graph[v]]
        best_clique = branch_and_bound(graph, new_candidates, new_clique, best_clique, start_time, time_limit)
    return best_clique

def main():
    test_files = [
        "hamming8-4.clq"
    ]

    total_time_limit = 7200.0

    for filename in test_files:
        print("Testing file:", filename)
        graph = read_dimacs(filename)
        overall_start = time.time()

        heuristic_start = time.time()
        initial_clique = heuristic_max_clique(graph, iterations=100)
        heuristic_time = time.time() - heuristic_start

        bnb_time_limit = max(0, total_time_limit - heuristic_time)
        bnb_start = time.time()
        candidates = list(range(len(graph)))
        best_clique = initial_clique.copy()
        best_clique = branch_and_bound(graph, candidates, set(), best_clique, bnb_start, bnb_time_limit)
        bnb_time = time.time() - bnb_start

        final_clique = best_clique
        clique_vertices = sorted(list(final_clique))

        print(len(final_clique))
        print(" ".join(str(v + 1) for v in clique_vertices))
        print(f"{heuristic_time:.2f}sec {bnb_time:.2f}sec")
        print("-----")

        overall_time = time.time() - overall_start
        print(f"Total time for {filename}: {overall_time:.2f}sec\n")

if __name__ == "__main__":
    main()

Testing file: hamming8-4.clq
16
1 16 52 61 86 91 103 106 151 154 166 171 196 205 241 256
0.11sec 32.95sec
-----
Total time for hamming8-4.clq: 33.07sec



In [8]:
import time
import random

def read_dimacs(filename):
    graph = []
    with open(filename, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or line[0] == 'c':
                continue
            parts = line.split()
            if parts[0] == 'p':
                n = int(parts[2])
                graph = [set() for _ in range(n)]
            elif parts[0] == 'e':
                u = int(parts[1]) - 1
                v = int(parts[2]) - 1
                graph[u].add(v)
                graph[v].add(u)
    return graph

def greedy_clique_from(graph, start):
    clique = {start}
    candidates = list(graph[start])
    random.shuffle(candidates)
    for v in candidates:
        if all(v in graph[u] for u in clique):
            clique.add(v)
            candidates = [w for w in candidates if w in graph[v]]
    return clique

def heuristic_max_clique(graph, iterations=100):
    n = len(graph)
    best_clique = set()
    vertices = list(range(n))
    random.shuffle(vertices)
    for v in vertices:
        clique = greedy_clique_from(graph, v)
        if len(clique) > len(best_clique):
            best_clique = clique
    for _ in range(iterations):
        v = random.choice(vertices)
        clique = greedy_clique_from(graph, v)
        if len(clique) > len(best_clique):
            best_clique = clique
    return best_clique

def greedy_coloring(graph, candidates):
    n = len(candidates)
    colors = [0] * n
    max_color = 0
    for i, v in enumerate(candidates):
        forbidden = set()
        for j in range(i):
            u = candidates[j]
            if u in graph[v]:
                forbidden.add(colors[j])
        col = 1
        while col in forbidden:
            col += 1
        colors[i] = col
        max_color = max(max_color, col)
    return max_color

def branch_and_bound(graph, candidates, current_clique, best_clique, start_time, time_limit):
    if time.time() - start_time >= time_limit:
        return best_clique
    if not candidates:
        if len(current_clique) > len(best_clique):
            return current_clique.copy()
        return best_clique

    color_bound = greedy_coloring(graph, candidates)
    if len(current_clique) + color_bound <= len(best_clique):
        return best_clique

    for i, v in enumerate(candidates):
        if time.time() - start_time >= time_limit:
            break
        new_clique = current_clique.copy()
        new_clique.add(v)
        new_candidates = [w for w in candidates[i+1:] if w in graph[v]]
        best_clique = branch_and_bound(graph, new_candidates, new_clique, best_clique, start_time, time_limit)
    return best_clique

def main():
    test_files = [
        "MANN_a9.clq"
    ]

    total_time_limit = 7200.0

    for filename in test_files:
        print("Testing file:", filename)
        graph = read_dimacs(filename)
        overall_start = time.time()

        heuristic_start = time.time()
        initial_clique = heuristic_max_clique(graph, iterations=100)
        heuristic_time = time.time() - heuristic_start

        bnb_time_limit = max(0, total_time_limit - heuristic_time)
        bnb_start = time.time()
        candidates = list(range(len(graph)))
        best_clique = initial_clique.copy()
        best_clique = branch_and_bound(graph, candidates, set(), best_clique, bnb_start, bnb_time_limit)
        bnb_time = time.time() - bnb_start

        final_clique = best_clique
        clique_vertices = sorted(list(final_clique))

        print(len(final_clique))
        print(" ".join(str(v + 1) for v in clique_vertices))
        print(f"{heuristic_time:.2f}sec {bnb_time:.2f}sec")
        print("-----")

        overall_time = time.time() - overall_start
        print(f"Total time for {filename}: {overall_time:.2f}sec\n")

if __name__ == "__main__":
    main()

Testing file: MANN_a9.clq
16
5 6 8 9 11 13 16 19 22 25 28 33 34 37 40 44
0.01sec 0.03sec
-----
Total time for MANN_a9.clq: 0.05sec



In [11]:
import time
import random

def read_dimacs(filename):
    graph = []
    with open(filename, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or line[0] == 'c':
                continue
            parts = line.split()
            if parts[0] == 'p':
                n = int(parts[2])
                graph = [set() for _ in range(n)]
            elif parts[0] == 'e':
                u = int(parts[1]) - 1
                v = int(parts[2]) - 1
                graph[u].add(v)
                graph[v].add(u)
    return graph

def greedy_clique_from(graph, start):
    clique = {start}
    candidates = list(graph[start])
    random.shuffle(candidates)
    for v in candidates:
        if all(v in graph[u] for u in clique):
            clique.add(v)
            candidates = [w for w in candidates if w in graph[v]]
    return clique

def heuristic_max_clique(graph, iterations=100):
    n = len(graph)
    best_clique = set()
    vertices = list(range(n))
    random.shuffle(vertices)
    for v in vertices:
        clique = greedy_clique_from(graph, v)
        if len(clique) > len(best_clique):
            best_clique = clique
    for _ in range(iterations):
        v = random.choice(vertices)
        clique = greedy_clique_from(graph, v)
        if len(clique) > len(best_clique):
            best_clique = clique
    return best_clique

def greedy_coloring(graph, candidates):
    n = len(candidates)
    colors = [0] * n
    max_color = 0
    for i, v in enumerate(candidates):
        forbidden = set()
        for j in range(i):
            u = candidates[j]
            if u in graph[v]:
                forbidden.add(colors[j])
        col = 1
        while col in forbidden:
            col += 1
        colors[i] = col
        max_color = max(max_color, col)
    return max_color

def branch_and_bound(graph, candidates, current_clique, best_clique, start_time, time_limit):
    if time.time() - start_time >= time_limit:
        return best_clique
    if not candidates:
        if len(current_clique) > len(best_clique):
            return current_clique.copy()
        return best_clique

    color_bound = greedy_coloring(graph, candidates)
    if len(current_clique) + color_bound <= len(best_clique):
        return best_clique

    for i, v in enumerate(candidates):
        if time.time() - start_time >= time_limit:
            break
        new_clique = current_clique.copy()
        new_clique.add(v)
        new_candidates = [w for w in candidates[i+1:] if w in graph[v]]
        best_clique = branch_and_bound(graph, new_candidates, new_clique, best_clique, start_time, time_limit)
    return best_clique

def main():
    test_files = [
        "sanr200_0.9.clq"
    ]

    total_time_limit = 7200.0

    for filename in test_files:
        print("Testing file:", filename)
        graph = read_dimacs(filename)
        overall_start = time.time()

        heuristic_start = time.time()
        initial_clique = heuristic_max_clique(graph, iterations=100)
        heuristic_time = time.time() - heuristic_start

        bnb_time_limit = max(0, total_time_limit - heuristic_time)
        bnb_start = time.time()
        candidates = list(range(len(graph)))
        best_clique = initial_clique.copy()
        best_clique = branch_and_bound(graph, candidates, set(), best_clique, bnb_start, bnb_time_limit)
        bnb_time = time.time() - bnb_start

        final_clique = best_clique
        clique_vertices = sorted(list(final_clique))

        print(len(final_clique))
        print(" ".join(str(v + 1) for v in clique_vertices))
        print(f"{heuristic_time:.2f}sec {bnb_time:.2f}sec")
        print("-----")

        overall_time = time.time() - overall_start
        print(f"Total time for {filename}: {overall_time:.2f}sec\n")

if __name__ == "__main__":
    main()

Testing file: sanr200_0.9.clq
39
1 2 7 15 25 26 27 50 57 63 66 67 68 84 93 94 98 101 104 107 110 116 118 119 120 124 125 143 149 150 153 160 164 166 167 175 187 190 200
0.10sec 7199.90sec
-----
Total time for sanr200_0.9.clq: 7200.00sec

